# M16 · 09 — B0: stage B size and budget from the A1 scaling runs

> **⚠ NOT EXECUTED.** Stage B is out of M16's scope (decision 0025, amendment of 2026-11-15).
> This notebook was written so the work can be resumed later. It has **never been run**, and there
> are **no results** in it. Do not read anything here as a measurement.

B0 (0025 §1): fit loss-vs-tokens on the three stage A scaling points (0.5B, 1B = `ablation-a1b-lr2e-3`,
2B) plus A2's 5B, extrapolate to 20–30B tokens at ~330M, and propose a budget for the owner (C2).
The measured points themselves are in notebook 02 and NIGHT_LOG; this notebook only fits them.

In [ ]:
import json
from pathlib import Path
from llm import HOME
RUNS, SFT, EVAL = HOME / 'runs', HOME / 'data' / 'sft', HOME / 'eval'
def load(p): return json.loads(Path(p).read_text())
def metrics(run): return [json.loads(l) for l in (RUNS / run / 'metrics.jsonl').read_text().splitlines()]
import numpy as np
pts = {}
for run in ('scaling-0.5b', 'ablation-a1b-lr2e-3', 'scaling-2b', 'stage-a-110m'):
    s = load(RUNS / run / 'status.json')
    pts[s['tokens']] = s['val_bpb']
D = np.array(sorted(pts)); L = np.array([pts[d] for d in D])
print(dict(zip(D, L)))

In [ ]:
# L(D) = E + B / D^beta, fitted on the 110M points (data-scaling at fixed size).
from scipy.optimize import curve_fit
f = lambda d, E, B, beta: E + B / (d / 1e9) ** beta
(E, B, beta), _ = curve_fit(f, D, L, p0=(0.8, 0.3, 0.5), maxfev=20000)
print(f'E={E:.4f} B={B:.4f} beta={beta:.3f}')
for d in (10e9, 20e9, 25e9, 30e9):
    print(f'110M at {d/1e9:.0f}B tokens: bpb {f(d, E, B, beta):.4f}')

Size step 110M → 330M: with only one model size measured, the size term cannot be fitted; B0
would use the A1 depth/width runs as a sanity check and a Chinchilla-style prior for it, and
report the uncertainty to the owner. Wall clock at 18.6k tok/s (notebook 00):

In [ ]:
for d in (20e9, 25e9, 30e9):
    print(f'{d/1e9:.0f}B tokens: {d / 18_600 / 86400:.1f} days of GPU')